# Cityscapes: полный запуск эксперимента в Google Colab

Notebook только настраивает окружение и вызывает скрипты проекта. Dataset, обучение, метрики и оценка остаются в `src/`. Cityscapes загружается через `kagglehub.dataset_download("electraawais/cityscape-dataset")`. Перед запуском выберите **Runtime → Change runtime type → GPU**.

## 1. Проверка GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU не найден. Выберите Runtime → Change runtime type → GPU.")
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
!nvidia-smi

## 2. Подключение Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import kagglehub

path = kagglehub.dataset_download("electraawais/cityscape-dataset")
print('KaggleHub dataset:', path)

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/atikkhon/diploma.git"
CITYSCAPES_ROOT = Path(path).expanduser().resolve()

PROJECT_DIR = Path('/content/Diploma')
DRIVE_ROOT = Path('/content/drive/MyDrive/cityscapes_robustness')
LOG_DIR = DRIVE_ROOT / 'logs'
STAGE_DIR = DRIVE_ROOT / 'stage_markers'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
TRAINING_METRICS_DIR = DRIVE_ROOT / 'training_metrics'
MLFLOW_DB = DRIVE_ROOT / 'mlflow.db'
MLFLOW_ARTIFACT_DIR = DRIVE_ROOT / 'mlartifacts'
SPLIT_MANIFEST = DRIVE_ROOT / 'split_manifest.csv'
RUNTIME_CONFIG = DRIVE_ROOT / 'experiment_colab.yaml'

for directory in (DRIVE_ROOT, LOG_DIR, STAGE_DIR, CHECKPOINT_DIR, TRAINING_METRICS_DIR, MLFLOW_ARTIFACT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

os.environ.update({
    'REPO_URL': REPO_URL,
    'PROJECT_DIR': str(PROJECT_DIR),
    'DRIVE_ROOT': str(DRIVE_ROOT),
    'LOG_DIR': str(LOG_DIR),
    'STAGE_DIR': str(STAGE_DIR),
    'CHECKPOINT_DIR': str(CHECKPOINT_DIR),
    'SPLIT_MANIFEST': str(SPLIT_MANIFEST),
    'RUNTIME_CONFIG': str(RUNTIME_CONFIG),
    'MLFLOW_TRACKING_URI': f'sqlite:///{MLFLOW_DB}',
    'MLFLOW_ARTIFACT_ROOT': MLFLOW_ARTIFACT_DIR.resolve().as_uri(),
})
print('Drive artifacts:', DRIVE_ROOT)
print('MLflow tracking URI:', os.environ['MLFLOW_TRACKING_URI'])
print('MLflow artifact root:', os.environ['MLFLOW_ARTIFACT_ROOT'])

## 3. Клонирование или обновление Git-репозитория

При повторном запуске выполняется `git pull --ff-only`; локальные изменения в `/content/Diploma` не перезаписываются.

In [ ]:
%%bash
set -euo pipefail
if [[ -d "$PROJECT_DIR/.git" ]]; then
  git -C "$PROJECT_DIR" pull --ff-only 2>&1 | tee -a "$LOG_DIR/git_update.log"
else
  git clone "$REPO_URL" "$PROJECT_DIR" 2>&1 | tee -a "$LOG_DIR/git_update.log"
fi

## 4. Установка `requirements.txt`

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -m pip install -r requirements.txt 2>&1 | tee -a "$LOG_DIR/install.log"

### Проверка SQLite MLflow

In [ ]:
import sys
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from src.tracking import check_mlflow_connection

mlflow_info = check_mlflow_connection('cityscapes_robustness')
print(mlflow_info)

## 5. Пути Cityscapes и Colab-конфигурация

Manifest, checkpoint, история обучения, логи и MLflow переживают разрыв runtime, потому что сохраняются в Google Drive.

In [ ]:
import sys
import yaml

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from src.dataset import discover_cityscapes_layout, prepare_train_id_masks

layout = discover_cityscapes_layout(CITYSCAPES_ROOT)
# Маски читаются на каждой эпохе, поэтому держим кэш на локальном SSD Colab.
# После обрыва runtime эта ячейка быстро восстановит его из Kaggle labelIds.
prepared_gtfine = Path('/content/prepared_gtFine')
train_masks = prepare_train_id_masks(
    layout['train_masks'], prepared_gtfine / 'train'
)
val_masks = prepare_train_id_masks(
    layout['val_masks'], prepared_gtfine / 'val'
)

with (PROJECT_DIR / 'configs/experiment.yaml').open(encoding='utf-8') as file:
    config = yaml.safe_load(file)
config['data']['root'] = str(CITYSCAPES_ROOT)
config['data']['train_images'] = str(layout['train_images'])
config['data']['train_masks'] = str(train_masks)
config['data']['official_val_images'] = str(layout['val_images'])
config['data']['official_val_masks'] = str(val_masks)
config['data']['split_file'] = str(SPLIT_MANIFEST)
config['training']['checkpoint_dir'] = str(CHECKPOINT_DIR)
config['training']['history_dir'] = str(TRAINING_METRICS_DIR)
config['training']['log_interval'] = 25
config['evaluation']['metrics_dir'] = str(PROJECT_DIR / 'outputs/metrics')
config['evaluation']['predictions_dir'] = str(PROJECT_DIR / 'outputs/predictions')
with RUNTIME_CONFIG.open('w', encoding='utf-8') as file:
    yaml.safe_dump(config, file, allow_unicode=True, sort_keys=False)
print('Runtime config:', RUNTIME_CONFIG)
print('Train images:', layout['train_images'])
print('Train masks:', train_masks)
print('Official val images:', layout['val_images'])
print('Official val masks:', val_masks)

## 6. Создание `split_manifest.csv`

Manifest пересоздаётся детерминированно, чтобы в нём всегда были актуальные локальные пути. При повторном запуске полная проверка масок не дублируется.

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
if [[ -s "$SPLIT_MANIFEST" ]]; then
  echo "Обновление путей существующего manifest без повторной полной проверки масок" | tee -a "$LOG_DIR/create_split.log"
  python scripts/create_split.py --config "$RUNTIME_CONFIG" --skip-mask-validation 2>&1 | tee -a "$LOG_DIR/create_split.log"
else
  python scripts/create_split.py --config "$RUNTIME_CONFIG" 2>&1 | tee -a "$LOG_DIR/create_split.log"
fi

## 7. Запуск тестов

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -m pytest tests/test_pipeline.py -v 2>&1 | tee -a "$LOG_DIR/pytest.log"

## 8. Dataset smoke test

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python scripts/smoke_test_dataset.py --config "$RUNTIME_CONFIG" --split dev 2>&1 | tee -a "$LOG_DIR/dataset_smoke_test.log"

In [ ]:
from IPython.display import Image, display
smoke_png = PROJECT_DIR / 'outputs/figures/dataset_smoke_test.png'
if not smoke_png.is_file():
    raise FileNotFoundError(f'Smoke-test PNG не найден: {smoke_png}')
display(Image(filename=str(smoke_png)))

## 9. Последовательное обучение baseline-моделей

Модели запускаются обычными независимыми командами в фиксированном порядке.

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/train_baselines.py --config "$RUNTIME_CONFIG" --models unet 2>&1 | tee -a "$LOG_DIR/train_unet.log"

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
print('GPU cache очищен после U-Net')

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/train_baselines.py --config "$RUNTIME_CONFIG" --models deeplabv3plus 2>&1 | tee -a "$LOG_DIR/train_deeplabv3plus.log"

In [ ]:
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
print('GPU cache очищен после DeepLabV3+')

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
python -u scripts/train_baselines.py --config "$RUNTIME_CONFIG" --models pspnet 2>&1 | tee -a "$LOG_DIR/train_pspnet.log"

In [ ]:
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
print('GPU cache очищен после PSPNet')

## 10. Clean evaluation

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
grep -q "if __name__" scripts/evaluate_clean.py || { echo "evaluate_clean.py ещё не реализован" >&2; exit 1; }
if [[ -f "$STAGE_DIR/evaluate_clean.done" ]]; then
  echo "Clean evaluation уже завершён."
else
  python scripts/evaluate_clean.py --config "$RUNTIME_CONFIG" 2>&1 | tee -a "$LOG_DIR/evaluate_clean.log"
  touch "$STAGE_DIR/evaluate_clean.done"
fi

## 11. Corruption evaluation

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
grep -q "if __name__" scripts/evaluate_corruptions.py || { echo "evaluate_corruptions.py ещё не реализован" >&2; exit 1; }
if [[ -f "$STAGE_DIR/evaluate_corruptions.done" ]]; then
  echo "Corruption evaluation уже завершён."
else
  python scripts/evaluate_corruptions.py --config "$RUNTIME_CONFIG" --corruptions configs/corruptions.yaml 2>&1 | tee -a "$LOG_DIR/evaluate_corruptions.log"
  touch "$STAGE_DIR/evaluate_corruptions.done"
fi

## 12. Robust training

Перед запуском укажите лучшую baseline-архитектуру в `robust_training.model_name` внутри `experiment_colab.yaml`. Выбор должен опираться только на internal validation CSV.

In [ ]:
# Укажите модель только после сравнения фактического dev mIoU в CSV.
BEST_MODEL = None  # 'unet', 'deeplabv3plus' или 'pspnet'

with RUNTIME_CONFIG.open(encoding='utf-8') as file:
    robust_config = yaml.safe_load(file)
if BEST_MODEL is not None:
    robust_config['robust_training']['model_name'] = BEST_MODEL
    with RUNTIME_CONFIG.open('w', encoding='utf-8') as file:
        yaml.safe_dump(robust_config, file, allow_unicode=True, sort_keys=False)
best_model = robust_config.get('robust_training', {}).get('model_name')
if best_model not in {'unet', 'deeplabv3plus', 'pspnet'}:
    raise ValueError("Укажите robust_training.model_name в experiment_colab.yaml после сравнения dev mIoU.")
print('Robust model:', best_model)

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
grep -q "if __name__" scripts/train_robust.py || { echo "train_robust.py ещё не реализован" >&2; exit 1; }
if [[ -f "$STAGE_DIR/train_robust.done" ]]; then
  echo "Robust training уже завершён."
else
  python scripts/train_robust.py --config "$RUNTIME_CONFIG" --corruptions configs/corruptions.yaml 2>&1 | tee -a "$LOG_DIR/train_robust.log"
  touch "$STAGE_DIR/train_robust.done"
fi

In [ ]:
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
print('GPU cache очищен после robust training')

## 13. Итоговые таблицы и графики

In [ ]:
%%bash
set -euo pipefail
cd "$PROJECT_DIR"
grep -q "if __name__" scripts/build_report_assets.py || { echo "build_report_assets.py ещё не реализован" >&2; exit 1; }
python scripts/build_report_assets.py --config "$RUNTIME_CONFIG" 2>&1 | tee -a "$LOG_DIR/build_report_assets.log"

## 14. Копирование outputs и checkpoints в Google Drive

In [ ]:
%%bash
set -euo pipefail
mkdir -p "$DRIVE_ROOT/final/outputs" "$DRIVE_ROOT/final/checkpoints"
rsync -a "$PROJECT_DIR/outputs/" "$DRIVE_ROOT/final/outputs/" 2>&1 | tee -a "$LOG_DIR/copy_results.log"
rsync -a "$CHECKPOINT_DIR/" "$DRIVE_ROOT/final/checkpoints/" 2>&1 | tee -a "$LOG_DIR/copy_results.log"
cp "$RUNTIME_CONFIG" "$DRIVE_ROOT/final/experiment_colab.yaml"
echo "Результаты сохранены в $DRIVE_ROOT/final"

## Повторный запуск после обрыва runtime

1. Снова выполните секции 1–5: подключите Drive, обновите репозиторий, установите зависимости и пересоздайте runtime-конфигурацию.
2. Manifest будет быстро обновлён с актуальными локальными путями; повторная полная проверка масок не выполняется.
3. Запускайте только нужную модель её обычной ячейкой. Незавершённая модель начнёт свои 8 эпох заново.
4. Clean/corruption/robust этапы используют `.done`-маркеры в `stage_markers`. Чтобы принудительно повторить этап, удалите только соответствующий marker-файл.
5. stdout каждого этапа дописывается в `Google Drive/cityscapes_robustness/logs`.